In [3]:
import requests
from bs4 import BeautifulSoup
import time

In [4]:
import pandas as pd
import csv

In [3]:
url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200"  # Mexico page
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')

In [4]:
print("=== Title ===")
print(soup.title.text if soup.title else 'No title')

=== Title ===

        Conflictos Mineros en México        (Observatorio de Conflictos Mineros de América Latina / OCMAL):
    


In [5]:
print("\n=== All Tables on Page ===")
tables = soup.find_all('table')
print(f"Found {len(tables)} tables")


=== All Tables on Page ===
Found 1 tables


In [6]:
print("\n=== First 1000 chars of first table (if exists) ===")
if tables:
    print(str(tables[0])[:1000])

#Each conflict is in a <tr> inside <tbody>, with data in <td> cells


=== First 1000 chars of first table (if exists) ===
<table cellpadding="0" cellspacing="0" class="table">
<thead>
<tr>
<th scope="col"><a href="/ocmal_db-v2/conflicto/lista/02024200?sort=nombre&amp;direction=asc">Nombre</a></th>
<th scope="col"><a href="/ocmal_db-v2/conflicto/lista/02024200?sort=lugar&amp;direction=asc">Lugar</a></th>
<th class="mobile-hide-column"><a href="/ocmal_db-v2/conflicto/lista/02024200?sort=inicio+da%C3%B1o&amp;direction=asc">Inicio Daño</a></th>
<th class="mobile-hide-column"><a href="/ocmal_db-v2/conflicto/lista/02024200?sort=inicio+conflicto&amp;direction=asc">Inicio Conflicto</a></th>
<th class="actions" scope="col">Detalles</th>
</tr>
</thead>
<tbody>
<tr>
<td>Campesinos del ejido Benito Juárez expulsan a minera Mag Silver</td>
<td>México (Ejido Benito Juárez Municipio de Buenaventura, Chihuahua )</td>
<td class="mobile-hide-column"></td>
<td class="mobile-hide-column">2012</td>
<td class="actions">
<a class="btn button" href="/ocmal_db-v2/conflicto/view

In [7]:
url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200"  # Mexico
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')

conflicts = []
rows = soup.select('tbody tr')  # All rows in table body

for row in rows:
    cells = row.find_all('td')
    if len(cells) >= 3:
        conflict = {
            'nombre': cells[0].get_text(strip=True),
            'lugar': cells[1].get_text(strip=True),
            'inicio_dano': cells[2].get_text(strip=True) if len(cells) > 2 else '',
            'inicio_conflicto': cells[3].get_text(strip=True) if len(cells) > 3 else ''
        }
        conflicts.append(conflict)

print(f"Found {len(conflicts)} conflicts in Mexico:")
print("-" * 60)
for i, c in enumerate(conflicts[:5], 1):  # Show first 5
    print(f"{i}. {c['nombre']} ({c['inicio_conflicto']})")

Found 10 conflicts in Mexico:
------------------------------------------------------------
1. Campesinos del ejido Benito Juárez expulsan a minera Mag Silver (2012)
2. Cananea, Grupo México y Sindicato Minero (2007)
3. Comuneros se oponen a la expansión de la mina Aquila (2000)
4. Comunidad indígena de Canoas se moviliza contra proyecto minero Rufus 5 en Colima (2019)
5. Comunidades campesinas frenan proyecto minero Cerro del Gallo en Guanajuato (2019)


In [9]:
url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200"
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')

print("=== All hrefs containing numbers or 'conflicto' ===")
links = soup.find_all('a', href=True)

for link in links:
    href = link['href']
    # Look for conflict IDs or numeric patterns
    if '/view/' in href or '/lista/' in href:
        print(href)

print(f"\n=== Total links found: {len(links)} ===")

# Also check for next/previous buttons
print("\n=== Check footer/body for 'siguiente', 'next', '1 2 3' ===")
body_text = soup.body.get_text()[:2000]  # Last part of page
print("Searching for pagination text...")

=== All hrefs containing numbers or 'conflicto' ===
mailto:?subject=Conflicto&body=https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200
https://twitter.com/intent/tweet?text=via @conflictominero: Conflicto&url=https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200
https://www.facebook.com/sharer/sharer.php?t=Conflicto&u=https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200
https://plus.google.com/share?title=Conflicto&url=https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200
/ocmal_db-v2/conflicto/lista/02024200?sort=nombre&direction=asc
/ocmal_db-v2/conflicto/lista/02024200?sort=lugar&direction=asc
/ocmal_db-v2/conflicto/lista/02024200?sort=inicio+da%C3%B1o&direction=asc
/ocmal_db-v2/conflicto/lista/02024200?sort=inicio+conflicto&direction=asc
/ocmal_db-v2/conflicto/view/952
/ocmal_db-v2/conflicto/view/39
/ocmal_db-v2/conflicto/view/245
/ocmal_db-v2/conflicto/view/1008
/ocmal_db-v2/conflicto/view/1007
/ocmal_db-v2

In [10]:
base_url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200"
all_conflicts = []

for page in range(1, 7):  # Pages 1 to 6
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}?page={page}"
    
    print(f"Fetching page {page}...")
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    rows = soup.select('tbody tr')
    
    for row in rows:
        cells = row.find_all('td')
        if len(cells) >= 3:
            conflict = {
                'nombre': cells[0].get_text(strip=True),
                'lugar': cells[1].get_text(strip=True),
                'inicio_dano': cells[2].get_text(strip=True) if len(cells) > 2 else '',
                'inicio_conflicto': cells[3].get_text(strip=True) if len(cells) > 3 else ''
            }
            all_conflicts.append(conflict)
    
    time.sleep(1)  # Wait 1 second between pages

print(f"\nTotal conflicts from Mexico: {len(all_conflicts)}")
for i, c in enumerate(all_conflicts[:10], 1):
    print(f"{i}. {c['nombre']} ({c['inicio_conflicto']})")

Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...
Fetching page 6...

Total conflicts from Mexico: 58
1. Campesinos del ejido Benito Juárez expulsan a minera Mag Silver (2012)
2. Cananea, Grupo México y Sindicato Minero (2007)
3. Comuneros se oponen a la expansión de la mina Aquila (2000)
4. Comunidad indígena de Canoas se moviliza contra proyecto minero Rufus 5 en Colima (2019)
5. Comunidades campesinas frenan proyecto minero Cerro del Gallo en Guanajuato (2019)
6. Comunidades de varios municipios  se movilizan contra contaminación de minera Autlán (1999)
7. Comunidades del municipio de  Ocotlán reclaman ilegalidad en concesiones mineras (2002)
8. Comunidades del Soconusco (Chiapas) Se movilizan contra la actividad minera (2015)
9. Comunidades Indígenas de Sierra Sur unidas en la oposición a la minería de hierro (1999)
10. Contaminación con desechos tóxicos y ausencia de políticas ambientales en Nuevo Mercurio (1980)


In [13]:
base_url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista/02024200"
all_conflicts = []

for page in range(1, 7):
    url = base_url if page == 1 else f"{base_url}?page={page}"
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    for row in soup.select('tbody tr'):
        cells = row.find_all('td')
        if len(cells) >= 3:
            all_conflicts.append({
                'nombre': cells[0].get_text(strip=True),
                'lugar': cells[1].get_text(strip=True),
                'inicio_dano': cells[2].get_text(strip=True) if len(cells) > 2 else '',
                'inicio_conflicto': cells[3].get_text(strip=True) if len(cells) > 3 else ''
            })
    time.sleep(1)

# Save to CSV
with open('mexico_conflicts.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=all_conflicts[0].keys())
    writer.writeheader()
    writer.writerows(all_conflicts)

print(f"✅ Saved {len(all_conflicts)} conflicts to mexico_conflicts.csv")

✅ Saved 58 conflicts to mexico_conflicts.csv


In [14]:
# Country IDs from the site
countries = {
    'Argentina': '02031300',
    'Bolivia': '02031900',
    'Brasil': '02032000',
    'Chile': '02032300',
    'Colombia': '02032400',
    'Mexico': '02024200',
    'Peru': '02034800'
}

base_url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista"
all_conflicts = []

for country, country_id in countries.items():
    print(f"\n=== Scraping {country} (ID: {country_id}) ===")
    
    # First, get the page to find how many pages exist
    url = f"{base_url}/{country_id}"
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find max page number
    links = soup.find_all('a', href=True)
    pages = []
    for link in links:
        if 'page=' in link['href']:
            page_num = int(link['href'].split('page=')[1].split('&')[0])
            pages.append(page_num)
    
    max_page = max(pages) if pages else 1
    print(f"Pages: {max_page}")
    
    for page in range(1, max_page + 1):
        url = f"{base_url}/{country_id}" if page == 1 else f"{base_url}/{country_id}?page={page}"
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for row in soup.select('tbody tr'):
            cells = row.find_all('td')
            if len(cells) >= 3:
                all_conflicts.append({
                    'pais': country,
                    'nombre': cells[0].get_text(strip=True),
                    'lugar': cells[1].get_text(strip=True),
                    'inicio_conflicto': cells[3].get_text(strip=True) if len(cells) > 3 else ''
                })
        
        time.sleep(2)

# Save
with open('all_conflicts.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=all_conflicts[0].keys())
    writer.writeheader()
    writer.writerows(all_conflicts)

print(f"\n{'='*60}")
print(f"✅ Total conflicts across all countries: {len(all_conflicts)}")


=== Scraping Argentina (ID: 02031300) ===
Pages: 3

=== Scraping Bolivia (ID: 02031900) ===
Pages: 1

=== Scraping Brasil (ID: 02032000) ===
Pages: 3

=== Scraping Chile (ID: 02032300) ===
Pages: 5

=== Scraping Colombia (ID: 02032400) ===
Pages: 2

=== Scraping Mexico (ID: 02024200) ===
Pages: 6

=== Scraping Peru (ID: 02034800) ===
Pages: 5

✅ Total conflicts across all countries: 236


In [16]:
url = "https://mapa.conflictosmineros.net/ocmal_db-v2/"
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')

print("=== Looking for tables ===")
tables = soup.find_all('table')
print(f"Found {len(tables)} tables")

if tables:
    print("\n=== First table content (first 1500 chars) ===")
    print(str(tables[0])[:1500])

=== Looking for tables ===
Found 3 tables

=== First table content (first 1500 chars) ===
<table cellpadding="0" cellspacing="0" class="table" width="100%"><thead><tr><th scope="col">Categorías</th></tr></thead>
<tbody><tr></tr></tbody>
</table>


In [17]:
url = "https://mapa.conflictosmineros.net/ocmal_db-v2/"
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')

tables = soup.find_all('table')

print(f"Found {len(tables)} tables\n")

for i, table in enumerate(tables):
    print(f"=== Table {i+1} ===")
    table_str = str(table)[:800]
    print(table_str)
    print("-" * 60)
    
    # Also check for country links in this table
    links = table.find_all('a', href=True)
    print(f"Links in table: {[link['href'] for link in links[:5]]}\n")

Found 3 tables

=== Table 1 ===
<table cellpadding="0" cellspacing="0" class="table" width="100%"><thead><tr><th scope="col">Categorías</th></tr></thead>
<tbody><tr></tr></tbody>
</table>
------------------------------------------------------------
Links in table: []

=== Table 2 ===
<table cellpadding="0" cellspacing="0" class="table" width="100%">
<thead>
<tr>
<th scope="cdol">Información general</th>
<th scope="cdol" style="float:right">Número de casos</th>
</tr>
</thead>
<tbody>
<tr>
<td style="float:left"><a class="btn button" href="/ocmal_db-v2/conflicto" style="font-size:110%;font-weight:bold" title="Haga clic para ver detalles">Conflictos publicados</a></td>
<td><div style="float:right;width:50px;font-weight:bold">
<a class="btn button" href="/ocmal_db-v2/conflicto" style='float:right;width:50px;font-weight:bold;color:white;font-size:110%;font-family: "Helvetica Neue",Helvetica,Roboto,Arial,sans-serif;font-weight:bold' title="Haga clic para ver detalles">284</a></div></td>
</tr

In [18]:
# First, extract all country IDs from the main page
main_url = "https://mapa.conflictosmineros.net/ocmal_db-v2/"
response = requests.get(main_url, timeout=10)
soup = BeautifulSoup(response.text, 'html.parser')

# Find Table 3 (countries table)
tables = soup.find_all('table')
countries_table = tables[2]  # Third table

countries = {}
for row in countries_table.find_all('tr')[1:]:  # Skip header
    cells = row.find_all('td')
    if len(cells) >= 2:
        country_name = cells[0].get_text(strip=True)
        link = cells[1].find('a', href=True)
        if link:
            country_id = link['href'].split('/')[-1]
            countries[country_name] = country_id
            print(f"Added: {country_name} ({country_id})")

print(f"\nTotal countries: {len(countries)}")

# Now scrape all conflicts
base_url = "https://mapa.conflictosmineros.net/ocmal_db-v2/conflicto/lista"
all_conflicts = []

for country, country_id in countries.items():
    print(f"\n=== Scraping {country} ===")
    
    url = f"{base_url}/{country_id}"
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find total pages
    links = soup.find_all('a', href=True)
    pages = []
    for link in links:
        if 'page=' in link['href']:
            try:
                page_num = int(link['href'].split('page=')[1].split('&')[0])
                pages.append(page_num)
            except:
                pass
    
    max_page = max(pages) if pages else 1
    print(f"Pages: {max_page}")
    
    for page in range(1, max_page + 1):
        page_url = url if page == 1 else f"{url}?page={page}"
        response = requests.get(page_url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for row in soup.select('tbody tr'):
            cells = row.find_all('td')
            if len(cells) >= 3:
                all_conflicts.append({
                    'pais': country,
                    'nombre': cells[0].get_text(strip=True),
                    'lugar': cells[1].get_text(strip=True),
                    'inicio_conflicto': cells[3].get_text(strip=True) if len(cells) > 3 else ''
                })
        
        time.sleep(1.5)
    
    time.sleep(2)

# Save
with open('all_latin_america_conflicts.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=all_conflicts[0].keys())
    writer.writeheader()
    writer.writerows(all_conflicts)

print(f"\n{'='*60}")
print(f"✅ TOTAL: {len(all_conflicts)} conflicts saved to all_latin_america_conflicts.csv")

Added: Argentina (02031300)
Added: Bolivia (02031900)
Added: Brasil (02032000)
Added: Chile (02032300)
Added: Colombia (02032400)
Added: Costa Rica (02022500)
Added: Ecuador (02032900)
Added: El Salvador (02023000)
Added: Guatemala (02023600)
Added: Guayana Francesa (02033200)
Added: Honduras (02023900)
Added: México (02024200)
Added: Nicaragua (02024500)
Added: Panamá (02024600)
Added: Paraguay (02034700)
Added: Perú (02034800)
Added: República Dominicana (02012800)
Added: Trinidad y Tobago (02015500)
Added: Uruguay (02035900)
Added: Venezuela (02036000)

Total countries: 20

=== Scraping Argentina ===
Pages: 3

=== Scraping Bolivia ===
Pages: 1

=== Scraping Brasil ===
Pages: 3

=== Scraping Chile ===
Pages: 5

=== Scraping Colombia ===
Pages: 2

=== Scraping Costa Rica ===
Pages: 1

=== Scraping Ecuador ===
Pages: 1

=== Scraping El Salvador ===
Pages: 1

=== Scraping Guatemala ===
Pages: 1

=== Scraping Guayana Francesa ===
Pages: 1

=== Scraping Honduras ===
Pages: 1

=== Scraping

In [ ]:
#ANALISING

In [5]:
conflicts = pd.read_csv("/home/diego/Documents/0.IRONHACK/1.SEPTEMBER/Projects/3.Defenders/all_latin_america_conflicts.csv")

In [21]:
conflicts.head()

,pais,nombre,lugar,inicio_conflicto
0,Argentina,33 comunidades de Salinas grandes denuncian e...,"Argentina (salinas grandes, provincia de Jujuy...",2010.0
1,Argentina,Andacollo Gold contamina aguas en Neuquén,"Argentina (Andacollo, Neuquén)",NaN
2,Argentina,Bajo la Alumbrera acusada de contaminación,"Argentina (Distrito Hualfin, Departamento de B...",1997.0
3,Argentina,Chilecito y Famatina contra Barrick Gold,"Argentina (Chilecito y Famatina, Provincia de ...",2006.0
4,Argentina,Comunidades mapuches defienden el agua de Lonc...,Argentina (Neuquén),NaN


In [23]:
conflicts["pais"].value_counts()

pais
México                  58
Chile                   49
Perú                    46
Argentina               28
Brasil                  26
Colombia                19
Bolivia                 10
Guatemala               10
Ecuador                  9
Nicaragua                7
Panamá                   7
Honduras                 6
El Salvador              3
República Dominicana     3
Costa Rica               2
Venezuela                2
Guayana Francesa         1
Paraguay                 1
Trinidad y Tobago        1
Uruguay                  1
Name: count, dtype: int64

In [24]:
conflicts.isna().any()

pais                False
nombre              False
lugar               False
inicio_conflicto     True
dtype: bool

In [26]:
conflicts["inicio_conflicto"].isnull().value_counts()

inicio_conflicto
False    259
True      30
Name: count, dtype: int64

In [27]:
conflicts[conflicts["inicio_conflicto"].isnull()]

,pais,nombre,lugar,inicio_conflicto
1,Argentina,Andacollo Gold contamina aguas en Neuquén,"Argentina (Andacollo, Neuquén)",NaN
4,Argentina,Comunidades mapuches defienden el agua de Lonc...,Argentina (Neuquén),NaN
5,Argentina,Detienen a mina de oro y cobre en el Valle de ...,"Argentina (Valle de Uco, Mendoza)",NaN
6,Argentina,Esquel: movilización histórica para paralizar ...,"Argentina (Esquel, Chubut)",NaN
15,Argentina,Minería de oro en la línea sur de Río Negro,Argentina (Línea sur de Río Negro),NaN
16,Argentina,Movilizaciones en General Alvear suspenden pro...,Argentina (Mendoza),NaN
28,Bolivia,Bolivia nacionaliza fundidora Vinto de la suiz...,Bolivia (Oruro),NaN
34,Bolivia,Protestas por contaminación de la Mina Reserv...,Bolivia (Potosí),NaN
35,Bolivia,San Bartolomé afecta a comunidades aymaras,Bolivia (Potosí),NaN
38,Brasil,Ação popular tenta impedir atividades da Comp...,Brasil (Região Sudeste; Estado de Minas Gerais...,NaN


In [28]:
conflicts.duplicated().sum()

np.int64(0)

In [29]:
conflicts.duplicated(subset=["nombre"]).sum()

np.int64(5)

In [31]:
#5 values duplicated
conflicts[conflicts.duplicated(subset=['nombre', 'inicio_conflicto'], keep=False)]
#These five values are transborder conflicts, but counted twice, one per country

,pais,nombre,lugar,inicio_conflicto
12,Argentina,Incertidumbre del proyecto Pachón,"Argentina, Chile (San Juan)",2005.0
18,Argentina,"Pascua Lama, glaciares en peligro, Chile-Argen...","Chile, Argentina (Límite internacional chileno...",2000.0
64,Chile,Amenazados se encuentran Valle del Lluta y Can...,"Chile, Perú (Vilvani, Depto Tacna; General Lag...",2007.0
88,Chile,Incertidumbre del proyecto Pachón,"Argentina, Chile (San Juan)",2005.0
100,Chile,"Pascua Lama, glaciares en peligro, Chile-Argen...","Chile, Argentina (Límite internacional chileno...",2000.0
133,Costa Rica,Proyecto Crucitas atenta contra el ecosistema ...,"Costa Rica, Nicaragua (Frontera Costa Rica y N...",NaN
144,El Salvador,Guatemaltecos y salvadoreños se oponen a proye...,"Guatemala, El Salvador (Asunción Mita (Guatema...",2007.0
149,Guatemala,Guatemaltecos y salvadoreños se oponen a proye...,"Guatemala, El Salvador (Asunción Mita (Guatema...",2007.0
225,Nicaragua,Proyecto Crucitas atenta contra el ecosistema ...,"Costa Rica, Nicaragua (Frontera Costa Rica y N...",NaN
237,Perú,Amenazados se encuentran Valle del Lluta y Can...,"Chile, Perú (Vilvani, Depto Tacna; General Lag...",2007.0


In [32]:
conflicts["pais"].nunique() #Counting how many countries

20

In [6]:
#Defining a region map with country

conflicts['region'] = conflicts['pais'].map({
    'México': 'North America',
    'Costa Rica': 'Central America',
    'El Salvador': 'Central America',
    'Guatemala': 'Central America',
    'Honduras': 'Central America',
    'Nicaragua': 'Central America',
    'Panamá': 'Central America',
    'Argentina': 'South America',
    'Bolivia': 'South America',
    'Brasil': 'South America',
    'Chile': 'South America',
    'Colombia': 'South America',
    'Ecuador': 'South America',
    'Paraguay': 'South America',
    'Perú': 'South America',
    'Uruguay': 'South America',
    'Venezuela': 'South America',
    'Guayana Francesa': 'South America',
    'República Dominicana': 'Caribbean',
    'Trinidad y Tobago': 'Caribbean'
})

In [37]:
conflicts.head()

,pais,nombre,lugar,inicio_conflicto,region
0,Argentina,33 comunidades de Salinas grandes denuncian e...,"Argentina (salinas grandes, provincia de Jujuy...",2010.0,South America
1,Argentina,Andacollo Gold contamina aguas en Neuquén,"Argentina (Andacollo, Neuquén)",NaN,South America
2,Argentina,Bajo la Alumbrera acusada de contaminación,"Argentina (Distrito Hualfin, Departamento de B...",1997.0,South America
3,Argentina,Chilecito y Famatina contra Barrick Gold,"Argentina (Chilecito y Famatina, Provincia de ...",2006.0,South America
4,Argentina,Comunidades mapuches defienden el agua de Lonc...,Argentina (Neuquén),NaN,South America


In [38]:
conflicts.tail()

,pais,nombre,lugar,inicio_conflicto,region
284,República Dominicana,Vecinos de Cotuí contra proyecto Pueblo Viejo ...,"República Dominicana (Cotuí, provincia Sánchez...",NaN,Caribbean
285,Trinidad y Tobago,Fundición de aluminio es repudiada en Trinidad...,"Trinidad y Tobago (Otaheite, Puerto España)",2005.0,Caribbean
286,Uruguay,Aratirí. NO a la minería de hierro a cielo abi...,"Uruguay (Departamento de Durazno, Florida, Tre...",2010.0,South America
287,Venezuela,Explotación carbonífera contamina sierras de P...,"Venezuela (Sierra Perijá, Estado de Zulia)",2000.0,South America
288,Venezuela,Proyecto Brisas-Cristinas: amenaza con acabar ...,"Venezuela (Municipio Sifontes, Estado Bolívar)",2015.0,South America


In [39]:
south_america = conflicts[conflicts["region"] == "South America"]

In [40]:
south_america.head()

,pais,nombre,lugar,inicio_conflicto,region
0,Argentina,33 comunidades de Salinas grandes denuncian e...,"Argentina (salinas grandes, provincia de Jujuy...",2010.0,South America
1,Argentina,Andacollo Gold contamina aguas en Neuquén,"Argentina (Andacollo, Neuquén)",NaN,South America
2,Argentina,Bajo la Alumbrera acusada de contaminación,"Argentina (Distrito Hualfin, Departamento de B...",1997.0,South America
3,Argentina,Chilecito y Famatina contra Barrick Gold,"Argentina (Chilecito y Famatina, Provincia de ...",2006.0,South America
4,Argentina,Comunidades mapuches defienden el agua de Lonc...,Argentina (Neuquén),NaN,South America


In [43]:
print(len(south_america)) #192 conflicts in South America

192


In [44]:
print(south_america["pais"].value_counts()) #Look at the South American countries with conflicts

pais
Chile               49
Perú                46
Argentina           28
Brasil              26
Colombia            19
Bolivia             10
Ecuador              9
Venezuela            2
Guayana Francesa     1
Paraguay             1
Uruguay              1
Name: count, dtype: int64


In [45]:
conflicts["region"].value_counts() #South America is the region with more conflicts

region
South America      192
North America       58
Central America     35
Caribbean            4
Name: count, dtype: int64

In [7]:
conflicts.to_csv('all_latin_america_conflicts_WITH_REGION.csv', index=False)